# 3. Attention

**Goal:** see exactly what attention computes, and why masks matter.

```
scores = q @ k.T / sqrt(d_head)
scores = mask(scores)        # future tokens -> -inf
attn   = softmax(scores)
out    = attn @ v
```

## Setup

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd()
while not (REPO / "packages" / "deepseek_flash_v4_mini").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "packages"))
sys.path.insert(0, str(REPO))
import torch

torch.manual_seed(0)
print("repo root:", REPO.name)
print("torch:", torch.__version__)

repo root: deepseek_flash_lab
torch: 2.13.0+cu130


## 1. Manual attention on a tiny example

One head, four tokens, four dimensions. We write the formula by hand.

In [2]:
import math

T, d_head = 4, 4
q = torch.randn(T, d_head)
k = torch.randn(T, d_head)
v = torch.randn(T, d_head)

scores = q @ k.T / math.sqrt(d_head)
print("raw scores:")
print(scores.round(decimals=2))

raw scores:
tensor([[ 1.5000,  0.2000, -0.1000, -1.6300],
        [-2.0900, -2.1900, -0.0500,  1.8800],
        [ 1.0700,  1.3000,  0.4900, -0.7400],
        [-0.9100, -1.3500, -0.4400,  1.7000]])


## 2. Causal mask

Token `t` may only read tokens `<= t`. We set forbidden scores to `-inf`
before the softmax, so their probability becomes zero.

In [3]:
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
print("allowed (1 = allowed):")
print(mask.int())

masked = scores.masked_fill(~mask, float("-inf"))
attn = torch.softmax(masked, dim=-1)
print("attention weights (rows sum to 1):")
print(attn.round(decimals=3))
print("row sums:", attn.sum(dim=-1).tolist())

allowed (1 = allowed):
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
attention weights (rows sum to 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5250, 0.4750, 0.0000, 0.0000],
        [0.3550, 0.4470, 0.1980, 0.0000],
        [0.0590, 0.0380, 0.0950, 0.8070]])
row sums: [1.0, 1.0, 0.9999998807907104, 1.0]


In [4]:
out = attn @ v
print("output shape:", tuple(out.shape), "-> [T, d_head]")
print("first output vector:", out[0].round(decimals=3).tolist())

output shape: (4, 4) -> [T, d_head]
first output vector: [-0.6140000224113464, 0.03200000151991844, -0.49300000071525574, 0.24799999594688416]


**Notice:** row 0 can only attend to token 0, so its weight is `1.0`.
Row 3 can attend to all four tokens.

## 3. The ready-made modules

The lab provides full causal attention and sliding-window attention.

In [5]:
from deepseek_flash_v4_mini.model.attention import CausalSelfAttention, SlidingWindowAttention

B, T, d_model, n_heads, d_head = 1, 5, 16, 2, 8
x = torch.randn(B, T, d_model)

full = CausalSelfAttention(d_model, n_heads, d_head, use_rope=False)
window = SlidingWindowAttention(d_model, n_heads, d_head, window=2, use_rope=False)
# Copy weights so the ONLY difference is the mask.
window.load_state_dict(full.state_dict())

y_full = full(x)
y_win = window(x)
print("input :", tuple(x.shape))
print("full  :", tuple(y_full.shape))
print("window:", tuple(y_win.shape))
print("mean abs difference:", (y_full - y_win).abs().mean().item())

input : (1, 5, 16)
full  : (1, 5, 16)
window: (1, 5, 16)
mean abs difference: 0.06580501794815063


**Notice:** shapes match, but values differ because the sliding window blocks
older tokens. This is the core trade-off: less compute, less context.

## 4. Test causality experimentally

If attention is causal, changing the **last** token must not change earlier
outputs.

In [6]:
x2 = x.clone()
x2[0, -1] = x2[0, -1] + 10.0

out1 = full(x)
out2 = full(x2)

print("positions 0..3 changed?", (out1[0, :-1] - out2[0, :-1]).abs().max().item())
print("position 4 changed?    ", (out1[0, -1] - out2[0, -1]).abs().max().item())

positions 0..3 changed? 0.0
position 4 changed?     3.249950647354126


**Notice:** the first value is 0 and the second is large. Causality holds:
future tokens cannot leak into the past.

**Try it:** `docs/EXERCISES.md` exercise 3 asks you to break the mask and watch
the leak appear.